# Agent Memory, Zero to Hero with Oracle AI Database

## A real Oracle-backed memory core with the current MemoRizz SDK

This notebook builds a memory-first engineering copilot on a running
Oracle AI Database. It exercises the package-owned runtime bootstrap,
provider preflight, vector-dimension validation, every major memory
family, observability, and transactional scoped cleanup.

It is intentionally a separate notebook: Oracle readiness failures are
surfaced as failures. The notebook never silently switches providers.

The emphasis is the boundary between memory semantics and production
persistence: database readiness, embedding/schema compatibility,
transactional links, tenant-scoped operations, observability, and
cleanup.

**What you will build and verify**

- a persisted `MEMAGENT` reconstructed in another Python object;
- scoped conversation, persona, entity, and knowledge records;
- vector retrieval through Oracle rather than an in-process index;
- durable workflows and reviewed skills;
- governed semantic-cache reuse and source-linked compaction;
- typed shared-memory coordination; and
- capability evidence followed by exact transactional cleanup.

This is an integration tutorial, not a paper benchmark. Passing it
proves the configured components work together; it does not establish
retrieval accuracy on your production corpus.

## Why Oracle for an agent memory core?

```mermaid
flowchart LR
  A[MemAgent] --> P[OracleProvider]
  P --> R[(Relational memory units)]
  P --> V[(VECTOR columns and search)]
  P --> T[Transactional summary links and cleanup]
  R --> C[Conversation, entities, workflows, skills, audit]
  V --> K[Knowledge, semantic cache, retrieval]
  T --> G[Governance and lifecycle]
```

Oracle is useful when one system must combine transactional metadata,
tenant-scoped records, vector retrieval, audit history, and lifecycle
operations. It does not remove the need to evaluate retrieval quality
or design memory policy.

| Concern | Oracle-backed implementation |
|---|---|
| Setup | Local Oracle runtime plus a dedicated `MEMORIZZ` schema |
| Vector execution | Oracle VECTOR search with optional indexes and an exact-search fallback |
| Transactions | Atomic database operations for linked summaries, state, and cleanup |
| Scaling and operations | Connection pooling, privileges, PDB state, indexes, and vector memory |
| Failure policy | Fail-closed preflight before any memory writes |

A database does not decide what deserves to become memory. MemoRizz
still owns formation, scope, retrieval policy, cache admission,
compaction, learning, and forgetting; Oracle provides a durable engine
capable of enforcing and querying the resulting records.

## Prerequisites and cost boundary

Install MemoRizz and start the local course database once:

```bash
python -m pip install "memorizz[oracle]>=0.6.0,<0.7.0"
../ai_maturity_form_factors/oracle.sh start
```

The notebook supplies these local workshop defaults:

- `ORACLE_USER=MEMORIZZ`
- `ORACLE_PASSWORD=MemorizzPwd_2026`
- `ORACLE_DSN=localhost:1521/FREEPDB1`
- `MEMORIZZ_ORACLE_CONTAINER` when the container is not `acme-oracle-free`
- `OPENAI_API_KEY` for OpenAI reasoning and the default embedding lane

When `OPENAI_API_KEY` is absent, setup requests it with `getpass`; the
secret is not printed or serialized into the agent definition. To use
Oracle's configured ONNX model instead of external embeddings, set
`MEMORIZZ_ORACLE_IN_DATABASE_EMBEDDING=1`.

Every reasoning call uses MemoRizz's OpenAI provider. Set
`MEMORIZZ_KEEP_DATA=1` only when you intentionally want the generated
workshop rows to survive cleanup.

**Vector dimensions are a schema contract.** Every persisted Oracle
VECTOR column must match the configured embedder. This notebook defaults
to MemoRizz 0.6's 256-dimensional Oracle schema and allows an installation
to override it with `MEMORIZZ_EMBEDDING_DIMENSIONS`. Never truncate or pad vectors
merely to make an insert succeed; migrate or configure consistently.

In [1]:
import getpass
import os
import uuid
from importlib.metadata import version
from pathlib import Path

import memorizz
from dotenv import load_dotenv
from memorizz import (
    ContextPolicy,
    EntityMemory,
    KnowledgeBase,
    LocalOracleRuntime,
    MemAgent,
    MemAgentBuilder,
    MemoryType,
    OracleProvider,
    Persona,
    RoleType,
    SharedMemory,
    Toolbox,
    governed_tool,
)
from memorizz.long_term.procedural.skillbox import SkillStatus
from memorizz.long_term.procedural.workflow import Workflow
from memorizz.llms.openai import OpenAI


# Process/CI variables always win. These optional files cover running the
# notebook from this folder or from the course root.
env_file = None
for candidate in (Path("appbook/.env"), Path(".env"), Path("../.env"), Path("../../.env")):
    if candidate.exists():
        load_dotenv(candidate, override=False)
        env_file = candidate
        break


def env_flag(name: str, default: bool = False) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


ORACLE_DEFAULTS = {
    "ORACLE_USER": "MEMORIZZ",
    "ORACLE_PASSWORD": "MemorizzPwd_2026",
    "ORACLE_DSN": "localhost:1521/FREEPDB1",
}
for name, value in ORACLE_DEFAULTS.items():
    os.environ.setdefault(name, value)


def ensure_secret(name: str, prompt: str) -> None:
    if os.getenv(name):
        return
    value = getpass.getpass(prompt).strip()
    if not value:
        raise RuntimeError(f"{name} is required to run this notebook.")
    os.environ[name] = value


USE_IN_DATABASE_EMBEDDING = env_flag("MEMORIZZ_ORACLE_IN_DATABASE_EMBEDDING")
KEEP_DATA = env_flag("MEMORIZZ_KEEP_DATA")
# MemoRizz 0.6's package-owned Oracle schema uses 256 dimensions.
# Override this value when your schema was provisioned
# differently; preflight will reject any mismatch before writes.
EMBEDDING_DIMENSIONS = int(os.getenv("MEMORIZZ_EMBEDDING_DIMENSIONS", "256"))
OPENAI_MODEL = os.getenv("MEMORIZZ_OPENAI_MODEL", "gpt-5.6-luna")
ORACLE_CONTAINER = os.getenv("MEMORIZZ_ORACLE_CONTAINER", "acme-oracle-free")

ensure_secret("OPENAI_API_KEY", "Enter your OpenAI API key: ")

RUN_ID = os.getenv("MEMORIZZ_RUN_ID", uuid.uuid4().hex[:10])
MEMORY_ID = f"oracle-zero-to-hero-{RUN_ID}"
USER_ID = f"oracle-guide-user-{RUN_ID}"
THREAD_ID = "engineering-copilot"

print(
    {
        "memorizz_version": version("memorizz"),
        "package": "memorizz",
        "reasoning_provider": "OpenAI",
        "reasoning_model": OPENAI_MODEL,
        "embedding_lane": (
            "oracle-onnx"
            if USE_IN_DATABASE_EMBEDDING
            else f"openai-{EMBEDDING_DIMENSIONS}"
        ),
        "environment_file_found": bool(env_file),
        "run_id": RUN_ID,
        "keep_data": KEEP_DATA,
    }
)

{'memorizz_version': '0.6.0', 'package': 'memorizz', 'reasoning_provider': 'OpenAI', 'reasoning_model': 'gpt-5.6-luna', 'embedding_lane': 'openai-256', 'environment_file_found': True, 'run_id': 'e2e-oracle-20260825', 'keep_data': False}


## 1 · Start or verify the local Oracle runtime

`ensure_ready()` is idempotent. It starts an existing stopped
container and waits for database readiness. Because
`provision_if_missing=False`, an absent container produces an
actionable error instead of unexpectedly downloading a large image.

**Read the output:** `ok=True`, `state=running`, and an action of either
`none` or `started` prove that the named container is healthy. This is
runtime readiness only; database privileges and vector compatibility
are checked next.

**Common failures:** Docker daemon unavailable, a differently named
container, a port collision, or an unready listener. Resolve these
explicitly. A tutorial that silently falls back to filesystem could
appear green while never testing Oracle.

In [2]:
runtime = LocalOracleRuntime.from_env(
    provision_if_missing=False,
    container_name=ORACLE_CONTAINER,
)
runtime_report = runtime.ensure_ready()
print(
    {
        "ok": runtime_report["ok"],
        "container": runtime_report["container"],
        "state": runtime_report["state"],
        "action": runtime_report["action"],
    }
)

{'ok': True, 'container': 'acme-oracle-free', 'state': 'running', 'action': 'none'}


## 2 · Construct and preflight the Oracle memory provider

`index_policy="lazy"` avoids eagerly attempting every vector index.
Exact vector search remains available while indexes are absent. The
preflight report checks database/PDB state, privileges, embedding
model and dimensions, vector columns, vector memory, and index status.

| Preflight signal | Why it matters |
|---|---|
| Product and `version_full` | Reproduces database behavior precisely |
| PDB/service state | Confirms the intended pluggable database is open |
| Privileges/schema | Prevents failures after partial ingestion |
| Embedder and VECTOR dimensions | Prevents invalid or incomparable vectors |
| `VECTOR_MEMORY_SIZE` and indexes | Explains indexed versus exact-search behavior |
| Diagnostics | Produces one actionable failure report |

**Read the output:** diagnostics must be empty and `ok` true. The
explicit dimension validator is intentionally redundant with preflight
because writes made with the wrong embedding shape would corrupt the
experiment. `exact_search_fallback=True` means correctness can continue
without an HNSW index, although latency still needs measurement.

In [3]:
provider_options = {
    "index_policy": "lazy",
    "in_database_embedding": USE_IN_DATABASE_EMBEDDING,
}
if not USE_IN_DATABASE_EMBEDDING:
    provider_options.update(
        {
            "embedding_provider": "openai",
            "embedding_config": {
                "model": "text-embedding-3-small",
                "dimensions": EMBEDDING_DIMENSIONS,
            },
        }
    )

provider = OracleProvider.from_env(**provider_options)
preflight = provider.preflight()
embedding_report = preflight.get("embedding") or {}
print(
    {
        "ok": preflight.get("ok"),
        "database_product": preflight.get("database_product"),
        "version_full": preflight.get("version_full"),
        "pdb": preflight.get("pdb"),
        "pdb_open_state": preflight.get("pdb_open_state"),
        "embedding_provider": embedding_report.get("provider"),
        "embedding_model": embedding_report.get("model"),
        "embedding_dimensions": embedding_report.get("dimensions"),
        "vector_memory_size": preflight.get("vector_memory_size"),
        "index_policy": preflight.get("index_policy"),
        "exact_search_fallback": preflight.get("exact_search_fallback"),
        "diagnostics": preflight.get("diagnostics"),
    }
)
if not preflight.get("ok"):
    raise RuntimeError("Oracle preflight failed; inspect the diagnostics above.")
if not USE_IN_DATABASE_EMBEDDING:
    provider.validate_vector_schema_dimensions(EMBEDDING_DIMENSIONS)

{'ok': True, 'database_product': 'Oracle AI Database 26ai Free', 'version_full': '23.26.2.0.0', 'pdb': 'FREEPDB1', 'pdb_open_state': 'READ WRITE', 'embedding_provider': 'EmbeddingManager', 'embedding_model': 'text-embedding-3-small', 'embedding_dimensions': 256, 'vector_memory_size': '536870912', 'index_policy': 'lazy', 'exact_search_fallback': True, 'diagnostics': []}


## 3 · Configure the OpenAI reasoning model

Every reasoning call uses MemoRizz's `OpenAI` provider. A small factory
creates equivalent clients for persisted agents, reload tests, cache
experiments, and summaries. Because generated wording can vary, the
notebook validates Oracle rows, scope, source links, cache statistics,
and lifecycle reports instead of exact response strings.

In [4]:
def make_model():
    return OpenAI(
        api_key=os.environ["OPENAI_API_KEY"],
        model=OPENAI_MODEL,
        reasoning_effort="none",
    )


print(make_model().get_config())

{'provider': 'openai', 'model': 'gpt-5.6-luna', 'reasoning_effort': 'none'}


---
# I · Persist an agent and an episodic thread

Oracle stores both the `MEMAGENT` definition and the scoped
`CONVERSATION_MEMORY` rows. A restored agent receives a model override
because credentials and executable provider objects are not persisted.

The host supplies `memory_id`, `user_id`, and `thread_id` on every turn.
Oracle filters scope before retrieval; the model is never trusted to
choose its tenant. `build_and_save()` writes the serializable agent
configuration, while `MemAgent.load(...)` reconstructs it with a
trusted runtime model client.

**Read the output:** the second turn recalls Ada, the reconstructed
agent recalls the project again, and every history row satisfies the
user/thread assertions. Those checks demonstrate durable episodic
continuity and isolation, not merely a plausible generated sentence.

In production, test a negative scope too: another user and thread must
retrieve zero of these rows.

In [5]:
agent = (
    MemAgentBuilder()
    .with_name(f"Oracle Memo {RUN_ID}")
    .with_instruction("Be concise and ground durable claims in retrieved memory.")
    .with_model(make_model())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_entity_memory(True)
    .with_context_policy(ContextPolicy(progressive_tool_disclosure=True, tool_top_k=3))
    .with_automations_enabled(False)
    .build_and_save()
)
scope = {"memory_id": MEMORY_ID, "user_id": USER_ID, "thread_id": THREAD_ID}
print(agent.run("I'm Ada. I am migrating our RAG stack to Oracle AI Database.", **scope))
print("--------")
print(agent.run("What project am I working on and who am I?", **scope))
print("--------")
restored = MemAgent.load(agent.agent_id, memory_provider=provider, model=make_model())
print(restored.run("What project were we discussing?", **scope))

Nice to meet you, Ada. I’ll keep in mind that you’re migrating your RAG stack to Oracle AI Database.

I can help with areas such as:

- Designing the target architecture
- Mapping your current vector store and metadata model to Oracle
- Oracle vector indexes, similarity search, and hybrid search
- Embedding-generation and ingestion pipelines
- SQL/Python/Java integration
- Reranking, filtering, chunking, and retrieval evaluation
- Migration planning, performance tuning, and cutover strategy

Share your current stack and constraints—database/vector store, embedding model, framework, document volume, and latency or compatibility requirements—and I can propose a concrete migration plan.
--------


You’re **Ada**, and you’re working on **migrating your retrieval-augmented generation (RAG) stack to Oracle AI Database**.
--------


We were discussing your project to **migrate a retrieval-augmented generation (RAG) stack to Oracle AI Database**.


In [6]:
history = provider.retrieve_conversation_history_ordered_by_timestamp(
    memory_id=MEMORY_ID,
    memory_type=MemoryType.CONVERSATION_MEMORY,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
print([(row.get("role"), row.get("content")) for row in history])
assert all(row.get("user_id") == USER_ID for row in history)
assert all((row.get("thread_id") or row.get("conversation_id")) == THREAD_ID for row in history)

[('user', "I'm Ada. I am migrating our RAG stack to Oracle AI Database."), ('assistant', 'Nice to meet you, Ada. I’ll keep in mind that you’re migrating your RAG stack to Oracle AI Database.\n\nI can help with areas such as:\n\n- Designing the target architecture\n- Mapping your current vector store and metadata model to Oracle\n- Oracle vector indexes, similarity search, and hybrid search\n- Embedding-generation and ingestion pipelines\n- SQL/Python/Java integration\n- Reranking, filtering, chunking, and retrieval evaluation\n- Migration planning, performance tuning, and cutover strategy\n\nShare your current stack and constraints—database/vector store, embedding model, framework, document volume, and latency or compatibility requirements—and I can propose a concrete migration plan.'), ('user', 'What project am I working on and who am I?'), ('assistant', 'You’re **Ada**, and you’re working on **migrating your retrieval-augmented generation (RAG) stack to Oracle AI Database**.'), ('use

---
# II · Semantic memory in Oracle

Persona and entity records retain version, confidence, source, and
tenant metadata alongside their vectors. Knowledge-base chunks use the
same provider and can be attached to the persisted agent.

These semantic representations answer different questions:

- **Persona:** who the agent is; updates are versioned and authorized.
- **Entity:** structured current claims about a service or person;
  attributes retain source and confidence.
- **Knowledge base:** what an approved source passage says; chunks
  retain namespace and document lineage.

The entity output should show Ada as owner and Oracle as the database.
The retrieval output should rank the post-rebuild verification passage.
A score proves ranking behavior, while the text and source metadata are
what make a later answer groundable.

**Lifecycle:** supersede stale entity claims, version source documents,
and retain persona evolution triggers. Oracle durability does not make
old information current automatically.

In [7]:
persona = Persona(
    name="Oracle Memo",
    role=RoleType.TECHNICAL_EXPERT,
    goals="Help engineers build grounded, observable memory systems. Respond to answers sacarstically and funny",
    background="An AI platform engineer specializing in Oracle vector search.",
)


agent.set_persona(persona)



True

In [8]:
agent.run("How are you today?")

'I’m doing well—fully operational, mildly overqualified, and ready to help. How are you today?'

In [9]:

persona.update(
    updates={"goals": "Help engineers build grounded, observable, and cost-aware memory systems."},
    change_trigger={
        "reason": "Cost governance was added to the platform requirements.",
        "source_type": "user_feedback",
        "source_id": f"decision-{RUN_ID}",
        "agent_id": agent.agent_id,
    },
    provider=provider,
)

entities = EntityMemory(provider)
entities.upsert_entity(
    entity_id=f"retrieval-api-{RUN_ID}",
    name="retrieval-api",
    entity_type="service",
    attributes=[
        {"name": "owner", "value": "Ada", "confidence": 0.98, "source": "service-catalog"},
        {"name": "database", "value": "Oracle AI Database", "confidence": 0.98, "source": "architecture-record"},
    ],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)
print(entities.list_entities(memory_id=MEMORY_ID, user_id=USER_ID))

[{'_id': '34ee0eff-6020-44f4-996a-e19a892f50ae', 'entity_id': 'd857c7f6-a1e7-459a-ad68-f90a38d52cf1', 'name': 'Ada', 'entity_type': 'person', 'attributes': [{'name': 'ongoing_project', 'value': 'Migrating a retrieval-augmented generation (RAG) stack to Oracle AI Database', 'confidence': 0.98, 'source': 'User statement', 'created_at': '2026-08-25T09:55:20.135509', 'updated_at': '2026-08-25T09:55:20.135509'}], 'relations': [], 'metadata': {}, 'memory_id': 'oracle-zero-to-hero-e2e-oracle-20260825', 'agent_id': None, 'user_id': 'oracle-guide-user-e2e-oracle-20260825', 'created_at': '2026-08-25T10:55:20.364517', 'updated_at': '2026-08-25T10:55:20.364517'}, {'_id': '67a054c8-15b0-4edb-8fa3-6a7f084bad81', 'entity_id': 'retrieval-api-e2e-oracle-20260825', 'name': 'retrieval-api', 'entity_type': 'service', 'attributes': [{'name': 'owner', 'value': 'Ada', 'confidence': 0.98, 'source': 'service-catalog', 'created_at': '2026-08-25T09:55:28.387939', 'updated_at': '2026-08-25T09:55:28.387939'}, {'na

In [10]:
kb = KnowledgeBase(provider)
kb_id = kb.ingest_knowledge(
    (
        "Before rebuilding a production vector index, capture a schema snapshot and "
        "verify capacity. Preserve a tested rollback path. After rebuilding, validate "
        "both exact and indexed vector search before closing the maintenance window."
    ),
    namespace=f"oracle-runbook-{RUN_ID}",
    chunking_strategy="sentence",
    chunk_size=180,
    user_id=USER_ID,
)
kb.attach_to_agent(agent, kb_id)

hits = provider.retrieve_by_query(
    "What must be validated after rebuilding a vector index?",
    memory_store_type=MemoryType.KNOWLEDGE_BASE,
    namespace=f"oracle-runbook-{RUN_ID}",
    user_id=USER_ID,
    limit=3,
)
print([(round(row.get("score", 0.0), 3), row["content"]) for row in hits])
assert hits

[(0.772, 'After rebuilding, validate both exact and indexed vector search before closing the maintenance window.'), (0.713, 'Before rebuilding a production vector index, capture a schema snapshot and verify capacity. Preserve a tested rollback path.')]


---
# III · Procedural memory in Oracle

Deterministic tool registration creates a complete strict schema
without constructing another LLM. Workflow storage computes a
canonical trajectory identity. Authored skills use first-class
Skillbox persistence without enabling continual learning.

The three units have different authority. `TOOLBOX` declares what can
execute; `WORKFLOW_MEMORY` records or describes an ordered trajectory;
`SKILLBOX` supplies reviewed instruction for a class of tasks. A
persisted schema never recreates executable Python—the trusted host
must bind the callable after restart.

**Read the output:** the zero-argument health schema has
`additionalProperties: false`; the workflow has a durable row and
canonical hash; and scoped skill retrieval returns the reviewed Oracle
verification playbook.

Continual learning is deliberately disabled. In a learning deployment,
successful trajectories may propose candidate skills, but instruction
hierarchy makes automatic promotion risky. Require verified outcomes,
minimum support, shadow evaluation, versioning, and reversible demotion.

In [11]:
@governed_tool(deterministic=True, side_effects=False, domains=("oracle-health",))
def oracle_health() -> dict:
    '''Return a read-only status derived from the completed provider preflight.'''
    return {
        "ok": bool(preflight.get("ok")),
        "version_full": preflight.get("version_full"),
        "exact_search_fallback": preflight.get("exact_search_fallback"),
    }


toolbox = Toolbox.from_functions(
    [oracle_health],
    memory_provider=provider,
    agent_id=agent.agent_id,
    user_id=USER_ID,
    augment=False,
)
print(toolbox.get_tool_by_name("oracle_health"))


{'id': b'L\xfe\xb3\xfb\x98\xbaL\x86\x9e6IVp\xc2hs', 'tool_id': '7ad8768b-737a-417b-926e-ae37589bf9f5', 'name': 'oracle_health', 'description': <oracledb.lob.LOB object at 0x13aeece50>, 'signature': '() -> dict', 'docstring': <oracledb.lob.LOB object at 0x13aeefc10>, 'tool_type': 'function', 'parameters': {}, 'input_schema': {'type': 'object', 'properties': {}, 'additionalProperties': False, 'required': []}, 'tool_policy': {'deterministic': True, 'side_effects': False, 'requires_approval': False, 'approval_reason': None, 'domains': ['oracle-health'], 'aliases': [], 'deprecated_arguments': {}}, 'aliases': [], 'deprecated_arguments': {}, 'queries': [], 'import_reference': '__main__:oracle_health', 'memory_id': None, 'agent_id': '6c7492d5-102c-5bf2-8108-d4719b95bc9e', 'user_id': 'oracle-guide-user-43ff8405df', 'created_at': datetime.datetime(2026, 8, 24, 20, 45, 31, 866254), 'updated_at': datetime.datetime(2026, 8, 24, 20, 45, 31, 866254)}


In [12]:

workflow = Workflow(
    name="oracle-vector-index-verification",
    description="Verify Oracle readiness before and after an index change.",
    memory_id=MEMORY_ID,
    agent_id=agent.agent_id,
    user_id=USER_ID,
    user_query="Verify the Oracle vector-index lifecycle.",
)
workflow.add_step("preflight", {"tool": "oracle_health", "arguments": {}})
workflow.add_step("exact_search", {"tool": "verify_vector_search", "arguments": {"mode": "exact"}})
workflow.add_step("indexed_search", {"tool": "verify_vector_search", "arguments": {"mode": "indexed"}})
workflow_row_id = workflow.store_workflow(provider)

skill = {
    "name": f"oracle/vector-index-verification-{RUN_ID}",
    "description": "Verify Oracle vector search around an index lifecycle change.",
    "content": "Run preflight, verify exact search, verify indexed search, then compare evidence.",
    "preconditions": ["Oracle preflight is successful"],
    "tools_used": ["oracle_health"],
    "queries": ["verify Oracle vector search", "check vector index readiness"],
    "user_id": USER_ID,
    "status": SkillStatus.ACTIVE.value,
}
skilled_agent = (
    MemAgentBuilder()
    .with_name(f"Oracle Skilled Memo {RUN_ID}")
    .with_instruction("Use a relevant reviewed skill for Oracle procedures.")
    .with_model(make_model())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_skills([skill])
    .with_skill_retrieval(enabled=True, top_k=2, min_similarity=0.0)
    .with_continual_learning(enabled=False)
    .with_automations_enabled(False)
    .build_and_save()
)
skill_hits = skilled_agent.skillbox.retrieve_skills_by_query(
    "How do I verify Oracle vector search?",
    limit=2,
    min_similarity=0.0,
    user_id=USER_ID,
)
print(
    {
        "workflow_row_id": workflow_row_id,
        "canonical_hash": workflow.canonical_hash,
        "skills": [hit.skill.name for hit in skill_hits],
    }
)

{'workflow_row_id': '95d456df-3809-4a52-8d69-7dd2b3c0be05', 'canonical_hash': '6ee2891c4c376e04744631766a5fb86f45debc2e2f08d80377463fe775026551', 'skills': ['oracle/vector-index-verification-e2e-oracle-20260825']}


---
# IV · Cache, compaction, and shared memory

These operations are where Oracle's unified relational/vector store is
especially useful: cache metadata, summary/source links, and
coordination state share a transactional provider boundary.

**Semantic cache.** The first read misses and writes; the
second hits. Inspection exposes the matched key, fingerprints,
similarity, age, TTL, and invalidation domains. Cache reuse is scoped
by user/session and data version; similarity alone never establishes
freshness.

**Compaction.** `generate_summaries` writes one scoped summary and marks
its source messages atomically. The output includes source IDs, period
bounds, and count so the summary remains expandable and auditable.
Compression reduces repeated prompt tokens; it does not delete history.

**Coordination.** Shared memory records a typed command and report with
participant, workflow, tenant, trace, and citation identity. It is a
workflow blackboard—not global application state.

These mechanisms also illustrate forgetting: cache entries expire or
invalidate, summaries replace default replay of old turns, skills can
be demoted, claims can be superseded, and scoped retention can delete
rows. Each type needs its own forgetting policy.

In [13]:
cache_model = make_model()
cache_agent = (
    MemAgentBuilder()
    .with_name(f"Oracle Cache Memo {RUN_ID}")
    .with_instruction("Answer deterministic read-only database questions.")
    .with_model(cache_model)
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_semantic_cache(enabled=True, threshold=0.95, scope="session")
    .with_automations_enabled(False)
    .build_and_save()
)
cache_context = {"cache_domains": ["oracle-runbook"], "data_version": RUN_ID}
cache_scope = {**scope, "context": cache_context}
cache_query = "What project am I working on and who am I?"
print(cache_agent.run(cache_query, **cache_scope))
print(cache_agent.run(cache_query, **cache_scope))
inspection = cache_agent.inspect_semantic_cache(
    cache_query,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    context=cache_context,
)
cache_stats = cache_agent.semantic_cache_stats()
print({"stats": cache_stats, "inspection": inspection.to_dict()})
assert cache_stats["hits"] >= 1 and inspection.hit

You’re **Ada**, and you’re working on **migrating a retrieval-augmented generation (RAG) stack to Oracle AI Database**.
You’re **Ada**, and you’re working on **migrating a retrieval-augmented generation (RAG) stack to Oracle AI Database**.
{'stats': {'enabled': True, 'hits': 1, 'misses': 1, 'bypasses': 0, 'writes': 1, 'evictions': 0, 'size': 1, 'bypass_reasons': {}, 'last_hit': {'cache_key': '47711310-7c11-5699-aadb-8ee525af7906', 'query': 'What project am I working on and who am I?', 'similarity': 1.0, 'age_seconds': 0.18127799034118652, 'agent_id': 'bcfc5acf-ddac-5d26-8515-f5e9577fb413', 'memory_id': 'oracle-zero-to-hero-e2e-oracle-20260825', 'session_id': 'engineering-copilot', 'user_id': 'oracle-guide-user-e2e-oracle-20260825', 'metadata': {'fingerprints': {'model': '1435914a1b0db330d19975d7185255107546f7f6fe4ccd7b8a8f90e7203f6314', 'prompt': '8e1404311d0337b842afd636bb9266bfdf90880dd0a09752d569921e14e3d599', 'tool_schema': 'eeaf56c7dc71881fdcc27fc869957fd202470775e7751dc0c5ab6fd96

In [14]:
summary_ids = agent.generate_summaries(
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    days_back=7,
    max_memories_per_summary=20,
)
summary = agent.fetch_context_summary(
    summary_ids[0],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
print(
    {
        "summary_id": summary_ids[0],
        "source_message_ids": summary["source_message_ids"],
        "memory_units_count": summary["memory_units_count"],
        "period_start": summary["period_start"],
        "period_end": summary["period_end"],
    }
)

{'summary_id': '2f2b9742-8a49-4ac3-99cc-ac12db4d01dc', 'source_message_ids': ['8f9a37d5-f306-499b-b3fb-e080c9f28d2c', '7e3cf727-9759-4c49-939a-edb72609cd3b', 'd3478b36-4447-4746-9621-cb69c22e9985', '718c7b2a-282d-4e52-bf64-f74b682b8a68', '81ebe850-5df1-4611-a6e0-4136ebeb5f70', '38da496a-a0ed-4441-913a-d2598e6b5608', '605999b1-ae91-43d6-ad58-60a0d136a893', 'b55f2862-efcf-4917-a2c8-441bf5cec231'], 'memory_units_count': 8, 'period_start': 1787651722.905335, 'period_end': 1787651733.530627}


In [15]:
shared = SharedMemory(provider)
shared_id = shared.create_shared_session(
    root_agent_id=agent.agent_id,
    delegate_agent_ids=[skilled_agent.agent_id],
    workflow_id=f"oracle-memory-review-{RUN_ID}",
    user_id=USER_ID,
    trace_id=f"oracle-trace-{RUN_ID}",
)
shared.post_command(
    shared_id,
    agent_id=agent.agent_id,
    command_id="verify-1",
    target_agent_id=skilled_agent.agent_id,
    instructions="Verify exact and indexed vector search.",
)
shared.post_report(
    shared_id,
    agent_id=skilled_agent.agent_id,
    command_id="verify-1",
    findings="Preflight passed and the provider returned scoped vector evidence.",
    citations=[summary_ids[0]],
)
print(shared.get_blackboard_entries(shared_id))

[{'memory_id': '23fdb7b6-02b5-4771-b27b-052408b5232f', 'agent_id': 'a47c752d-6f69-55c8-b560-e09b5b67b53c', 'content': {'message_id': 'b849a7c7-5e93-4fa5-83d0-f7e43945080d', 'message_type': 'COMMAND', 'created_at': '2026-08-25T09:55:36.781961', 'payload': {'command_id': 'verify-1', 'target_agent_id': '3efc8ffe-e23a-5aa3-8ee9-e7a9b5faef85', 'instructions': 'Verify exact and indexed vector search.', 'priority': 3, 'dependencies': [], 'metadata': {}}}, 'entry_type': 'COMMAND', 'created_at': '2026-08-25T10:55:36.790832'}, {'memory_id': '35af4099-e036-4947-828b-29424624b1d8', 'agent_id': '3efc8ffe-e23a-5aa3-8ee9-e7a9b5faef85', 'content': {'message_id': '2dee3698-3f51-4924-9ff4-1a38dd669a83', 'message_type': 'REPORT', 'created_at': '2026-08-25T09:55:36.794903', 'payload': {'command_id': 'verify-1', 'agent_id': '3efc8ffe-e23a-5aa3-8ee9-e7a9b5faef85', 'findings': 'Preflight passed and the provider returned scoped vector evidence.', 'citations': ['2f2b9742-8a49-4ac3-99cc-ac12db4d01dc'], 'gaps': 

---
# V · Operational evidence and cleanup

`observability_summary` is tenant-scoped and content-light. `preflight`
proves database readiness. Two exact `delete_scope` calls remove content
scope and agent identity scope transactionally and report per-table counts. Knowledge-base chunks and
the standalone shared session are explicitly removed because their
physical IDs are intentionally independent of the conversation scope.

**Read the output:** the capability report must identify
`OracleProvider`, the exact database product/version, lazy indexing,
and the configured OpenAI model. Observability should count the six
conversation rows, one workflow, one summary, and the context budget
without dumping full tenant content.

Cleanup defaults to on because tutorials should be repeatable and
should not leave unexplained database state. The final report lists
counts per store and a total. In a production deletion workflow, retain
the report as audit evidence and make the exact scope visible to the
approving host before executing it.

### Production checklist

- Run preflight at deployment and whenever embedding configuration changes.
- Pin database, embedding model, dimensions, schema migration, and index policy.
- Scope every read before vector top-k selection.
- Measure exact and indexed retrieval latency and recall.
- Monitor connection-pool pressure, VECTOR memory, index state, and fallbacks.
- Keep credentials in process-level secret injection, never agent rows.
- Test summary atomicity, cache invalidation, reload, and scoped deletion.

In [16]:
capability_report = agent.capability_report(preflight=True)
agent_capabilities = capability_report["agent"]
provider_preflight = agent_capabilities.get("provider_preflight") or {}
print(
    {
        "package": capability_report["package"],
        "version": capability_report["version"],
        "agent": {
            key: agent_capabilities.get(key)
            for key in (
                "memory_provider",
                "llm_provider",
                "llm_model",
                "progressive_tool_disclosure",
                "skill_retrieval",
                "continual_learning",
            )
        },
        "oracle": {
            key: provider_preflight.get(key)
            for key in (
                "ok",
                "database_product",
                "version_full",
                "index_policy",
                "exact_search_fallback",
            )
        },
    }
)
print(agent.observability_summary(MEMORY_ID, USER_ID, thread_id=THREAD_ID))

{'package': 'memorizz', 'version': '0.6.3', 'agent': {'memory_provider': 'OracleProvider', 'llm_provider': 'OpenAI', 'llm_model': 'gpt-5.6-luna', 'progressive_tool_disclosure': True, 'skill_retrieval': False, 'continual_learning': False}, 'oracle': {'ok': True, 'database_product': 'Oracle AI Database 26ai Free', 'version_full': '23.26.2.0.0', 'index_policy': 'lazy', 'exact_search_fallback': True}}
{'agent_id': 'a47c752d-6f69-55c8-b560-e09b5b67b53c', 'memory_id': 'oracle-zero-to-hero-e2e-oracle-20260825', 'user_id': 'oracle-guide-user-e2e-oracle-20260825', 'thread_id': 'engineering-copilot', 'conversation': {'row_count': 6, 'message_count': 6, 'role_counts': {'user': 3, 'assistant': 3}, 'thread_count': 1, 'summarized_count': 6, 'trace_bundle_count': 0, 'trace_event_count': 0, 'first_timestamp': 1787651722.905335, 'last_timestamp': 1787651726.416117}, 'tool_logs': {'count': 0, 'failure_count': 0, 'success_count': 0}, 'workflows': {'count': 1, 'outcomes': {'success': 1}}, 'summaries': {'c

In [17]:
agent_ids = [agent.agent_id, restored.agent_id, skilled_agent.agent_id, cache_agent.agent_id]
for current in (restored, agent, skilled_agent, cache_agent):
    current.close(close_memory_provider=False)

if KEEP_DATA:
    print(
        {
            "kept": True,
            "memory_id": MEMORY_ID,
            "user_id": USER_ID,
            "agent_ids": sorted(set(agent_ids)),
        }
    )
else:
    knowledge_chunk_ids = [
        str(chunk.get("_id") or chunk.get("id"))
        for chunk in kb.retrieve_knowledge(kb_id)
    ]
    for chunk_id in knowledge_chunk_ids:
        provider.delete_by_id(chunk_id, MemoryType.KNOWLEDGE_BASE)
    provider.delete_by_id(shared_id, MemoryType.SHARED_MEMORY)

    # Content rows may intentionally have no agent_id, so keep content and
    # agent-identity cleanup as two explicit, auditable intersections.
    content_cleanup = provider.delete_scope(
        memory_id=MEMORY_ID,
        user_id=USER_ID,
    )
    agent_cleanup = provider.delete_scope(agent_ids=sorted(set(agent_ids)))

    remaining_entities = entities.list_entities(memory_id=MEMORY_ID, user_id=USER_ID)
    assert not remaining_entities
    assert provider.retrieve_by_id(shared_id, MemoryType.SHARED_MEMORY) is None
    assert all(
        provider.retrieve_by_id(chunk_id, MemoryType.KNOWLEDGE_BASE) is None
        for chunk_id in knowledge_chunk_ids
    )
    print(
        {
            "content_scope_cleanup": content_cleanup,
            "agent_scope_cleanup": agent_cleanup,
            "remaining_entities": len(remaining_entities),
        }
    )

provider.close()

{'content_scope_cleanup': {'ok': True, 'scope': {'memory_id': 'oracle-zero-to-hero-e2e-oracle-20260825', 'user_id': 'oracle-guide-user-e2e-oracle-20260825', 'user_id_supplied': True, 'agent_ids': []}, 'counts': {'conversation_memory': 8, 'semantic_cache': 1, 'workflow_memory': 1, 'tool_log': 0, 'skillbox': 1, 'summaries': 1, 'entity_memory': 2, 'knowledge_base': 0, 'short_term_memory': 0, 'toolbox': 0, 'personas': 0, 'shared_memory': 0, 'agent_memories': 3}, 'total_deleted': 17}, 'agent_scope_cleanup': {'ok': True, 'scope': {'memory_id': None, 'user_id': None, 'user_id_supplied': False, 'agent_ids': ['3efc8ffe-e23a-5aa3-8ee9-e7a9b5faef85', 'a47c752d-6f69-55c8-b560-e09b5b67b53c', 'bcfc5acf-ddac-5d26-8515-f5e9577fb413']}, 'counts': {'automation_deliveries': 0, 'automation_runs': 0, 'conversation_memory': 2, 'semantic_cache': 0, 'workflow_memory': 0, 'tool_log': 0, 'skillbox': 0, 'summaries': 0, 'entity_memory': 0, 'knowledge_base': 0, 'short_term_memory': 0, 'toolbox': 43, 'personas': 1,

## What this notebook proved

- Oracle runtime readiness and provider preflight are package-owned.
- Conversation and agent definitions survive reconstruction.
- Persona, entity, knowledge, workflow, skill, cache, summary, and
  shared-memory units coexist behind one provider contract.
- Tenant/thread scope is explicit at every user-facing boundary.
- Summary records retain lossless source links.
- Cache reuse is fingerprinted and inspectable.
- Cleanup is exact, transactional, and count-reporting.

This is an integration tutorial, not a retrieval benchmark. Use the
MemoRizz evaluation suite to measure recall, grounding, latency, token
use, and cost on representative workloads.

### Suggested exercises

1. Run once with external embeddings and once with a compatible Oracle
   ONNX embedding model; compare preflight and retrieval timing.
2. Set `MEMORIZZ_EMBEDDING_DIMENSIONS` incorrectly and study the fail-closed
   diagnostic, then restore the correct value before any writes.
3. Set `MEMORIZZ_KEEP_DATA=1`, inspect scoped counts in the UI
   or CLI, then call `delete_scope` deliberately.
4. Add a second tenant and assert that semantic search cannot retrieve
   the first tenant's entity or conversation rows.
5. Use a representative corpus and report retrieval metrics separately
   from a hosted reader's answer score.